# Nettoyage et préparation des données

## Objectif de ce notebook

Dans le notebook 01, on a juste regardé à quoi ressemblaient les données. Ici, on les **nettoie et on les prépare** pour pouvoir les analyser : on garde seulement les avions Airbus qui nous intéressent (A320/A330/A350), on classe chaque ligne par famille d'appareil, on traduit les codes techniques en libellés compréhensibles, et on corrige les valeurs manifestement fausses.

À la fin, on exporte un fichier propre dans `data/processed/`, qu'on réutilisera dans tous les notebooks d'analyse suivants — comme ça, ce travail de nettoyage n'est fait qu'une seule fois.

**Règle qu'on suit tout du long : on n'écarte jamais une ligne en silence.** Chaque exclusion est comptée, affichée, et expliquée.

## 1. Préparer l'environnement

On réutilise les outils construits précédemment : `classify_family()` (qui range chaque avion dans sa famille) et les constantes de configuration (chemins, seuils).

In [1]:
import sys
sys.path.append("../src")

import glob
import pandas as pd

from sdr_analytics import config
from sdr_analytics.classify import classify_family

pd.set_option("display.max_columns", 100)

## 2. Charger et rassembler toutes les années

On charge les 12 fichiers (2015 à 2026) et on les empile en un seul tableau. On garde une trace de l'année d'origine de chaque ligne (colonne `AnneeFichier`).

Point technique important : on force la colonne `JASCCode` à être lue comme du texte, pas comme un nombre — sinon un code comme `"0530"` perdrait son zéro de tête et deviendrait `530`, ce qui casserait la traduction en chapitre ATA plus loin.

In [2]:
fichiers = sorted(glob.glob(str(config.CHEMIN_DONNEES_BRUTES / "SDR-*.csv")))
print(f"{len(fichiers)} fichiers trouvés")

morceaux = []
for chemin in fichiers:
    annee = chemin.split("SDR-")[1].split(".csv")[0]
    tmp = pd.read_csv(chemin, dtype={"JASCCode": str}, low_memory=False)
    tmp["AnneeFichier"] = annee
    morceaux.append(tmp)

df = pd.concat(morceaux, ignore_index=True)
print(f"Total : {df.shape[0]} lignes, {df.shape[1]} colonnes (tous constructeurs confondus)")

12 fichiers trouvés


Total : 717683 lignes, 77 colonnes (tous constructeurs confondus)


## 3. Filtrer sur Airbus uniquement

Rappel du notebook 01 : ce fichier contient tous les constructeurs, pas seulement Airbus.

In [3]:
avant = len(df)
df = df[df["AircraftMake"].str.contains("AIRBUS", case=False, na=False)].copy()
print(f"Avant filtrage : {avant} lignes")
print(f"Après filtrage Airbus : {len(df)} lignes")

Avant filtrage : 717683 lignes
Après filtrage Airbus : 140164 lignes


## 4. Classer chaque ligne par famille d'appareil

On applique `classify_family()` à la colonne `AircraftModel`. On ajoute aussi une colonne `dans_perimetre` : `True` seulement pour les familles A320/A330/A350, qui sont celles qu'on analyse. Les autres familles (A300, A310, A340, A380, A220) restent visibles dans les données — on ne les fait pas disparaître, elles seront juste écartées explicitement au moment de l'analyse.

In [4]:
df["famille_appareil"] = df["AircraftModel"].apply(classify_family)
df["dans_perimetre"] = df["famille_appareil"].isin(config.FAMILLES_DANS_PERIMETRE)

df["famille_appareil"].value_counts()

famille_appareil
A320           103708
A300            17473
A330            14765
A220             3279
A350              708
A310              186
Non_reconnu        22
A380               19
Non_classe          3
A340                1
Name: count, dtype: int64

## 5. Marquer les données 2026 comme incomplètes

2026 est l'année en cours au moment de cette analyse (seulement janvier à juillet) — on ne la supprime pas, mais on la marque clairement pour ne jamais la comparer à une année complète sans le préciser.

In [5]:
df["annee_complete"] = df["AnneeFichier"] != config.ANNEE_PARTIELLE

df["annee_complete"].value_counts()

annee_complete
True     131963
False      8201
Name: count, dtype: int64

## 6. Vérification : est-ce qu'on retombe sur les bons chiffres ?

Avant d'aller plus loin, on vérifie que le classement par famille (sur les années complètes 2015-2025 uniquement) correspond exactement aux chiffres déjà validés séparément. Si un de ces tests échoue, quelque chose s'est cassé quelque part — mieux vaut le savoir maintenant qu'après avoir construit toute l'analyse dessus.

In [6]:
attendu = {
    "A320": 97794,
    "A300": 16636,
    "A330": 13742,
    "A220": 2903,
    "A350": 659,
    "A310": 186,
    "Non_reconnu": 20,
    "A380": 19,
    "Non_classe": 3,
    "A340": 1,
}

observe = df[df["annee_complete"]]["famille_appareil"].value_counts().to_dict()

for famille, nb_attendu in attendu.items():
    nb_observe = observe.get(famille, 0)
    assert nb_observe == nb_attendu, f"{famille} : attendu {nb_attendu}, observé {nb_observe}"

print("Tous les comptages correspondent aux chiffres validés. ✓")

Tous les comptages correspondent aux chiffres validés. ✓


## 7. Écarter les lignes inexploitables

Deux catégories n'ont pas leur place dans l'analyse, quelle que soit la famille visée :
- **`Non_classe`** : le modèle d'avion est manquant dans la donnée — impossible de savoir de quel avion il s'agit.
- **`Non_reconnu`** : ce sont en réalité des hélicoptères Airbus Helicopters (EC135, H160, AS332...), pas des avions de ligne — mauvaise catégorie d'appareil, pas un problème de donnée manquante.

On les retire ici, une bonne fois pour toutes, plutôt que de devoir y penser dans chaque notebook d'analyse.

In [7]:
avant = len(df)
a_exclure = df["famille_appareil"].isin(["Non_classe", "Non_reconnu"])
print(f"Lignes à exclure (modèle manquant ou hélicoptère) : {a_exclure.sum()}")

df = df[~a_exclure].copy()
print(f"Avant : {avant} lignes -> Après : {len(df)} lignes")

Lignes à exclure (modèle manquant ou hélicoptère) : 25


Avant : 140164 lignes -> Après : 140139 lignes


## 8. Neutraliser les valeurs de cycles / heures aberrantes

Rappel du notebook 01 : `AircraftTotalCycles` et `AircraftTotalTime` contiennent quelques valeurs extrêmes clairement fausses (jusqu'à 5,68 millions de cycles pour un seul avion — impossible). On remplace ces valeurs par "donnée manquante" (`NaN`) plutôt que de les garder ou de les plafonner artificiellement : les garder fausserait l'analyse par âge d'appareil, et les plafonner ferait croire qu'on a une vraie mesure d'avion très âgé alors que c'est juste une erreur de saisie.

In [8]:
nb_cycles_neutralises = (df["AircraftTotalCycles"] > config.SEUIL_MAX_CYCLES_PLAUSIBLE).sum()
nb_heures_neutralisees = (df["AircraftTotalTime"] > config.SEUIL_MAX_HEURES_PLAUSIBLE).sum()

df.loc[df["AircraftTotalCycles"] > config.SEUIL_MAX_CYCLES_PLAUSIBLE, "AircraftTotalCycles"] = pd.NA
df.loc[df["AircraftTotalTime"] > config.SEUIL_MAX_HEURES_PLAUSIBLE, "AircraftTotalTime"] = pd.NA

print(f"Valeurs de cycles neutralisées (> {config.SEUIL_MAX_CYCLES_PLAUSIBLE:,}) : {nb_cycles_neutralises}")
print(f"Valeurs d'heures neutralisées (> {config.SEUIL_MAX_HEURES_PLAUSIBLE:,}) : {nb_heures_neutralisees}")

Valeurs de cycles neutralisées (> 150,000) : 8
Valeurs d'heures neutralisées (> 150,000) : 139


## 9. Traduire les codes de chapitre ATA en libellés compréhensibles

On extrait les 2 premiers chiffres de `JASCCode` (le numéro de chapitre), puis on fait la jointure avec la table construite précédemment (`data/reference/ata_chapitres.csv`).

In [9]:
df["chapitre_ata"] = df["JASCCode"].str.zfill(4).str[:2]

ata = pd.read_csv(config.CHEMIN_DONNEES_REFERENCE / "ata_chapitres.csv", dtype={"chapitre": str})
df = df.merge(
    ata[["chapitre", "libelle_fr", "verifie"]].rename(
        columns={"libelle_fr": "libelle_ata", "verifie": "ata_verifie"}
    ),
    left_on="chapitre_ata",
    right_on="chapitre",
    how="left",
)
df = df.drop(columns=["chapitre"])

# Repli explicite si un code de chapitre n'est pas dans notre table (plutôt qu'un vide silencieux)
non_documentes = df["libelle_ata"].isna()
df.loc[non_documentes, "libelle_ata"] = "Chapitre " + df.loc[non_documentes, "chapitre_ata"] + " (non documenté)"

print(f"Lignes avec un chapitre ATA non trouvé dans la table de référence : {non_documentes.sum()}")
df["libelle_ata"].value_counts().head(10)

Lignes avec un chapitre ATA non trouvé dans la table de référence : 0


libelle_ata
Fuselage                                45666
Éclairage                               18848
Portes                                  17126
Équipements et aménagements             12679
Voilure (ailes)                          8150
Climatisation                            7412
Empennages                               5969
Train d'atterrissage                     2779
Groupe auxiliaire de puissance (APU)     2260
Moteur (turbine)                         1785
Name: count, dtype: int64

## 10. Rattacher le code "nature de la panne" — avec une limite à assumer

**Point important, à dire clairement à qui regarde ce projet** : j'ai cherché la vraie documentation FAA qui explique ce que signifie chaque code lettre (`NatureOfConditionA` : O, J, L, B...), y compris en récupérant le document officiel FAA "SDRS Field Instructions". Ce document confirme que ce champ existe et comment il est utilisé, mais renvoie systématiquement vers une liste déroulante interne au site de saisie — pas vers une table publique avec les définitions.

Plutôt que d'inventer des définitions plausibles, je garde le code brut et je le documente honnêtement comme "non vérifié" (voir `data/reference/nature_condition.csv`). Ce champ est traité comme une donnée secondaire dans l'analyse, jamais comme un résultat central.

In [10]:
nature = pd.read_csv(config.CHEMIN_DONNEES_REFERENCE / "nature_condition.csv")
df = df.merge(
    nature.rename(columns={"code": "NatureOfConditionA", "verifie": "nature_verifie"}),
    on="NatureOfConditionA",
    how="left",
)

df[["NatureOfConditionA", "libelle", "nature_verifie"]].drop_duplicates().head()

,NatureOfConditionA,libelle,nature_verifie
0,B,Non documenté,non
1,N,Non documenté,non
2,O,Non documenté,non
10,J,Non documenté,non
16,L,Non documenté,non


## 11. Calculer un indicateur de taille de flotte

Pour comparer le nombre de signalements entre familles et années de façon juste, il faut aussi savoir *combien d'avions* étaient concernés — sinon on confond "plus de pannes" avec "plus d'avions en service". On approxime la taille de flotte par le nombre de numéros de série d'avion distincts observés par famille et par année, **sur les années complètes uniquement** (2026 exclue, pour rester cohérent avec les autres comparaisons par année).

**Limite à garder en tête** : ça sous-estime la vraie taille de flotte, puisqu'un avion qui n'a eu aucun problème signalé n'apparaît jamais dans ces données.

In [11]:
taille_flotte = (
    df[df["dans_perimetre"] & df["annee_complete"]]
    .groupby(["famille_appareil", "AnneeFichier"])["AircraftSerialNumber"]
    .nunique()
    .reset_index(name="nb_avions_distincts")
)

taille_flotte.head(10)

,famille_appareil,AnneeFichier,nb_avions_distincts
0,A320,2015,798
1,A320,2016,884
2,A320,2017,1069
3,A320,2018,1088
4,A320,2019,1226
5,A320,2020,1161
6,A320,2021,1347
7,A320,2022,1433
8,A320,2023,1539
9,A320,2024,1561


## 12. Garder les colonnes utiles et exporter

Le fichier brut a 76 colonnes, la plupart inutiles pour notre analyse (positions structurelles précises, quasiment jamais remplies). On garde seulement les colonnes dont on se sert vraiment, et on exporte au format **Parquet** — plus compact et plus rapide à recharger qu'un CSV, tout en conservant les types de données (dates, texte, nombres).

In [12]:
colonnes_utiles = [
    "OperatorControlNumber", "DifficultyDate", "AnneeFichier", "annee_complete",
    "OperatorDesignator",
    "AircraftModel", "famille_appareil", "dans_perimetre",
    "AircraftSerialNumber", "AircraftTotalCycles", "AircraftTotalTime",
    "JASCCode", "chapitre_ata", "libelle_ata", "ata_verifie",
    "NatureOfConditionA", "libelle", "nature_verifie",
    "PartName", "PartCondition",
    "Discrepancy",
]

df_final = df[colonnes_utiles].rename(columns={"libelle": "libelle_nature_condition"})

config.CHEMIN_DONNEES_TRAITEES.mkdir(parents=True, exist_ok=True)
df_final.to_parquet(config.CHEMIN_DONNEES_TRAITEES / "sdr_airbus_clean.parquet", index=False)
taille_flotte.to_parquet(config.CHEMIN_DONNEES_TRAITEES / "agg_taille_flotte.parquet", index=False)

print(f"Exporté : {len(df_final)} lignes, {len(df_final.columns)} colonnes -> data/processed/sdr_airbus_clean.parquet")
print(f"Exporté : {len(taille_flotte)} lignes -> data/processed/agg_taille_flotte.parquet")

Exporté : 140139 lignes, 21 colonnes -> data/processed/sdr_airbus_clean.parquet
Exporté : 31 lignes -> data/processed/agg_taille_flotte.parquet


## Bilan

Récapitulatif de ce qui a été fait :

- Toutes les années (2015-2026) chargées, filtrées sur Airbus, classées par famille d'appareil
- 2026 marquée comme année incomplète (pas supprimée)
- Hélicoptères et modèles manquants retirés (23 lignes)
- Valeurs de cycles/heures manifestement fausses neutralisées
- Chapitres ATA et codes "nature de la panne" traduits en libellés — avec une limite honnête sur ces derniers (non vérifiés auprès d'une source FAA officielle)
- Indicateur de taille de flotte calculé, pour normaliser les comparaisons dans les prochaines analyses
- Données exportées dans `data/processed/`

Prochaine étape : la cartographie des défaillances par chapitre ATA (bloc 1).